# [1.1] SC26 Tutorial — Session 1, Part 1: Introduction to MLIR (CPU)

This part compiles and runs a matrix-matrix product, `out[i, j] += sum_k a[i, k] * b[k, j]`, two ways:

1. Written by hand as `scf.for` loops, lowered to LLVM and JIT-executed.
2. Named by a single custom op, `blas.matmul`, and lowered to that same loop form by a compiler pass, written in Exercise 3.

For each exercise there is a **solved matrix-*vector* example** to use as a reference.

Everything in this part runs on CPU only — no GPU runtime needed.

**Running locally in VS Code instead?** See [`RUNNING_LOCALLY.md`](https://github.com/MaheshRavishankar/sc26_tutorial/blob/main/RUNNING_LOCALLY.md).

## [1.2] Setup

**Run the cells in order, starting here.** Later cells use names defined by earlier ones, so skipping ahead can lead to strange errors. To catch up after losing track, `Ctrl+F8` on Colab runs every cell above the current one.

The next cells download a prebuilt MLIR toolchain built for this tutorial: `mlir-opt`, `mlir-tblgen`, the MLIR Python bindings and CUDA runtime support. The [repo README](https://github.com/MaheshRavishankar/sc26_tutorial) explains why it is a custom build rather than the official LLVM release or a pip package.

In [163]:
# [1.3]
REPO = "MaheshRavishankar/sc26_tutorial"
RELEASE_TAG = "session1-toolchain-v1"
ASSET_NAME = "mlir-tutorial-llvm.tar.gz"

In [164]:
# [1.4]
import os

# Set MLIR_TUTORIAL_INSTALL_DIR to iterate locally (e.g. in VS Code) against
# an already-built toolchain -- skips the download entirely if it's already
# there. Unset in Colab, so this always defaults to the normal /content path.
INSTALL_DIR = os.environ.get("MLIR_TUTORIAL_INSTALL_DIR", "/content/mlir-tutorial-llvm")

if os.path.exists(f"{INSTALL_DIR}/bin/mlir-opt"):
    print(f"Using existing toolchain at {INSTALL_DIR}")
else:
    import requests

    tar_path = f"{INSTALL_DIR}.tar.gz"
    url = f"https://github.com/{REPO}/releases/download/{RELEASE_TAG}/{ASSET_NAME}"
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(tar_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)

    !mkdir -p {INSTALL_DIR}
    !tar xf {tar_path} -C {INSTALL_DIR} --strip-components=1

Using existing toolchain at /content/mlir-tutorial-llvm


In [165]:
# [1.5]
import glob, os, re, sys

os.environ["PATH"] = f"{INSTALL_DIR}/bin:" + os.environ["PATH"]
sys.path.insert(0, f"{INSTALL_DIR}/python_packages/mlir_core")

# The MLIR Python bindings only import into the Python version they were built
# for (3.13, as on Colab). Check that here: with any other version, the next
# cell fails with a confusing "cannot import name 'ir'".
bindings = glob.glob(f"{INSTALL_DIR}/python_packages/mlir_core/mlir/_mlir_libs/_mlir.cpython-*.so")
assert bindings, f"no MLIR Python bindings in {INSTALL_DIR}"
built_for = ".".join(re.search(r"cpython-(\d)(\d+)", bindings[0]).groups())
kernel = f"{sys.version_info.major}.{sys.version_info.minor}"
assert built_for == kernel, (
    f"The MLIR Python bindings are built for Python {built_for}; this kernel is "
    f"Python {kernel}. Switch to a Python {built_for} kernel.")

!mlir-opt --version

LLVM (http://llvm.org/):
  LLVM version 22.1.8
  Optimized build.


In [166]:
# [1.6]
from mlir.ir import Context, Module
from mlir.passmanager import PassManager
from mlir.execution_engine import ExecutionEngine
import mlir.runtime as rt
import ctypes
import numpy as np

print("MLIR Python bindings imported OK")

MLIR Python bindings imported OK


**[1.7]** The next cell clones the tutorial's repo.

In [167]:
# [1.8]
# Set MLIR_TUTORIAL_REPO_DIR to iterate locally against an already-cloned
# checkout -- skips the clone entirely if it's already there.
REPO_DIR = os.environ.get("MLIR_TUTORIAL_REPO_DIR", "/content/sc26_tutorial")
if os.path.isdir(REPO_DIR):
    print(f"Using existing repo checkout at {REPO_DIR}")
else:
    # --no-single-branch: every branch's tip, still at depth 1. It costs
    # nothing measurable, and it means the solution branches are already in
    # the checkout, for taking a solved file out of one.
    !git clone --depth 1 --no-single-branch https://github.com/{REPO} {REPO_DIR}

Using existing repo checkout at /content/sc26_tutorial


## [1.9] Step 1 — the loop nest, by hand

**Exercise 1: fill in the empty `@matmul` body of `session1/matmul_scf.mlir`** and save with Ctrl/Cmd+S. The cell below opens it in the editor, next to its reference.

**The reference is `session1/matvec_scf.mlir`,** a solved matrix-vector product. Matmul is the same loop nest with one more loop:

- an outer loop `%i` over the rows of `a`;
- inside it, `%j` over the columns of `b`;
- inside that, the same `iter_args` reduction as matvec, now over `%k`, with its result stored to `out[%i, %j]`.

**Like matvec, matmul adds to `out` (`+=`) instead of overwriting it.** So the reduction starts from `memref.load %out[%i, %j]`, not from zero.

The examples use fixed sizes (4x3 and so on) to keep them simple. The loops take their bounds from `memref.dim`, so they would work unchanged on sizes known only at run time (`memref<?x?xf32>`).

The four cells below are provided. Re-run them after each edit; they fail until the loop nest is right.

In [168]:
# [1.10]
# Opens the files below. On Colab, files.view() puts them in the editor
# pane -- use it instead of the file browser, which refuses some file
# types. Running locally there is no google.colab, so this prints the
# paths instead, to open in an editor.

try:
    from google.colab import files

    files.view(f"{REPO_DIR}/session1/matmul_scf.mlir")   # Exercise 1
    files.view(f"{REPO_DIR}/session1/matvec_scf.mlir")   # reference, complete
except ModuleNotFoundError:
    print("open in an editor:")
    print(f"    {REPO_DIR}/session1/matmul_scf.mlir   (Exercise 1)")
    print(f"    {REPO_DIR}/session1/matvec_scf.mlir   (reference, complete)")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**[1.11]** First, parse and re-print the file with `mlir-opt`. No pass pipeline is given, so this just round-trips it through MLIR's parser and printer — the quickest way to catch a syntax error or a type mismatch in the file.

In [169]:
# [1.12]
MATMUL_SCF_FILE = f"{REPO_DIR}/session1/matmul_scf.mlir"

!mlir-opt {MATMUL_SCF_FILE}
assert _exit_code == 0

module {
  func.func @matmul(%arg0: memref<4x3xf32>, %arg1: memref<3x5xf32>, %arg2: memref<4x5xf32>) {
    %c0 = arith.constant 0 : index
    %c1 = arith.constant 1 : index
    %dim = memref.dim %arg0, %c0 : memref<4x3xf32>
    %dim_0 = memref.dim %arg1, %c1 : memref<3x5xf32>
    %dim_1 = memref.dim %arg0, %c1 : memref<4x3xf32>
    scf.for %arg3 = %c0 to %dim step %c1 {
      scf.for %arg4 = %c0 to %dim_0 step %c1 {
        %0 = memref.load %arg2[%arg3, %arg4] : memref<4x5xf32>
        %1 = scf.for %arg5 = %c0 to %dim_1 step %c1 iter_args(%arg6 = %0) -> (f32) {
          %2 = memref.load %arg0[%arg3, %arg5] : memref<4x3xf32>
          %3 = memref.load %arg1[%arg5, %arg4] : memref<3x5xf32>
          %4 = arith.mulf %2, %3 : f32
          %5 = arith.addf %arg6, %4 : f32
          scf.yield %5 : f32
        }
        memref.store %1, %arg2[%arg3, %arg4] : memref<4x5xf32>
      }
    }
    return
  }
}



**[1.13]** Now parse it into an in-memory MLIR module on the Python side.

In [170]:
# [1.14]
ctx = Context()
module = Module.parseFile(MATMUL_SCF_FILE, ctx)
print(module)

module {
  func.func @matmul(%arg0: memref<4x3xf32>, %arg1: memref<3x5xf32>, %arg2: memref<4x5xf32>) {
    %c0 = arith.constant 0 : index
    %c1 = arith.constant 1 : index
    %dim = memref.dim %arg0, %c0 : memref<4x3xf32>
    %dim_0 = memref.dim %arg1, %c1 : memref<3x5xf32>
    %dim_1 = memref.dim %arg0, %c1 : memref<4x3xf32>
    scf.for %arg3 = %c0 to %dim step %c1 {
      scf.for %arg4 = %c0 to %dim_0 step %c1 {
        %0 = memref.load %arg2[%arg3, %arg4] : memref<4x5xf32>
        %1 = scf.for %arg5 = %c0 to %dim_1 step %c1 iter_args(%arg6 = %0) -> (f32) {
          %2 = memref.load %arg0[%arg3, %arg5] : memref<4x3xf32>
          %3 = memref.load %arg1[%arg5, %arg4] : memref<3x5xf32>
          %4 = arith.mulf %2, %3 : f32
          %5 = arith.addf %arg6, %4 : f32
          scf.yield %5 : f32
        }
        memref.store %1, %arg2[%arg3, %arg4] : memref<4x5xf32>
      }
    }
    return
  }
}



**[1.15]** Lower `scf` -> `cf` -> `llvm` in place. The cell prints the result, which is long, low-level LLVM-dialect IR: there is no need to read it. It is there to show that the loops are gone. `llvm-request-c-wrappers` makes `@matmul` callable from Python via `ExecutionEngine.invoke`.

In [171]:
# [1.16]
CPU_LOWERING_PIPELINE = (
    "builtin.module("
    "func.func(llvm-request-c-wrappers),"
    "convert-scf-to-cf,"
    "convert-cf-to-llvm,"
    "convert-arith-to-llvm,"
    "finalize-memref-to-llvm,"
    "convert-func-to-llvm,"
    "reconcile-unrealized-casts"
    ")"
)

PassManager.parse(CPU_LOWERING_PIPELINE, ctx).run(module.operation)
print(module)

module {
  llvm.func @matmul(%arg0: !llvm.ptr, %arg1: !llvm.ptr, %arg2: i64, %arg3: i64, %arg4: i64, %arg5: i64, %arg6: i64, %arg7: !llvm.ptr, %arg8: !llvm.ptr, %arg9: i64, %arg10: i64, %arg11: i64, %arg12: i64, %arg13: i64, %arg14: !llvm.ptr, %arg15: !llvm.ptr, %arg16: i64, %arg17: i64, %arg18: i64, %arg19: i64, %arg20: i64) attributes {llvm.emit_c_interface} {
    %0 = llvm.mlir.poison : !llvm.struct<(ptr, ptr, i64, array<2 x i64>, array<2 x i64>)>
    %1 = llvm.insertvalue %arg14, %0[0] : !llvm.struct<(ptr, ptr, i64, array<2 x i64>, array<2 x i64>)> 
    %2 = llvm.insertvalue %arg15, %1[1] : !llvm.struct<(ptr, ptr, i64, array<2 x i64>, array<2 x i64>)> 
    %3 = llvm.insertvalue %arg16, %2[2] : !llvm.struct<(ptr, ptr, i64, array<2 x i64>, array<2 x i64>)> 
    %4 = llvm.insertvalue %arg17, %3[3, 0] : !llvm.struct<(ptr, ptr, i64, array<2 x i64>, array<2 x i64>)> 
    %5 = llvm.insertvalue %arg19, %4[4, 0] : !llvm.struct<(ptr, ptr, i64, array<2 x i64>, array<2 x i64>)> 
    %6 = llvm.

**[1.17]** JIT-compile the lowered module and invoke it with numpy arrays, checking the result against `numpy`'s own matmul.

Until `matmul_scf.mlir` is filled in, this fails on an all-zero result rather than an error — an empty function body is perfectly valid MLIR, it just never writes anything. That is what "not done yet" looks like here.

In [172]:
# [1.18]
def run_and_check(module, entry_point, args, expected, shared_libs=None, needs_init=False):
    """JIT-compile `module` (already lowered to LLVM) and call
    `entry_point(*args)`. Each arg is a numpy array passed as a memref; by
    convention the last one is the output buffer the kernel writes into. It is
    zeroed first -- we reuse the same buffers across steps, and without this a
    kernel that writes nothing would pass on the previous step's result -- then
    checked against `expected` and returned."""
    args[-1][...] = 0

    ee = ExecutionEngine(module, shared_libs=shared_libs or [])
    if needs_init:
        ee.initialize()
    ee.invoke(
        entry_point,
        *[ctypes.pointer(ctypes.pointer(rt.get_ranked_memref_descriptor(a)))
          for a in args],
    )

    out = args[-1]
    print("out      =", out)
    print("expected =", expected)
    np.testing.assert_allclose(out, expected)
    print("OK")
    return out


a = np.arange(12, dtype=np.float32).reshape(4, 3)
b = np.arange(15, dtype=np.float32).reshape(3, 5)
out = np.zeros((4, 5), dtype=np.float32)

run_and_check(module, "matmul", [a, b, out], expected=a @ b)

out      = [[ 25.  28.  31.  34.  37.]
 [ 70.  82.  94. 106. 118.]
 [115. 136. 157. 178. 199.]
 [160. 190. 220. 250. 280.]]
expected = [[ 25.  28.  31.  34.  37.]
 [ 70.  82.  94. 106. 118.]
 [115. 136. 157. 178. 199.]
 [160. 190. 220. 250. 280.]]
OK


array([[ 25.,  28.,  31.,  34.,  37.],
       [ 70.,  82.,  94., 106., 118.],
       [115., 136., 157., 178., 199.],
       [160., 190., 220., 250., 280.]], dtype=float32)

## [1.19] Step 2 — the `blas` dialect

This step defines a custom op, `blas.matmul`, in a dialect called `blas`, and the pass that lowers it to the loops from step 1. The dialect and its passes are compiled into one plugin, `libBlasPlugin.so`, which `mlir-opt` loads at run time with `--load-dialect-plugin` and `--load-pass-plugin`.

**First, build the plugin as it is in the repo.** It already defines `blas.matvec`, the reference for this step. Exercise 2 adds `blas.matmul`. The first build takes 1–2 minutes.

In [173]:
# [1.20]
import shutil

# Colab has neither; skip the install where both are already on PATH (and
# where apt-get would need root, e.g. running locally).
if shutil.which("cmake") and shutil.which("ninja"):
    print("cmake and ninja are already installed")
else:
    !apt-get install -y -qq cmake ninja-build > /dev/null
    assert _exit_code == 0

cmake and ninja are already installed


In [174]:
# [1.21]
PLUGIN_SRC = f"{REPO_DIR}/session1/blas-dialect"
PLUGIN_BUILD = f"{PLUGIN_SRC}/build"
PLUGIN_SO = f"{PLUGIN_BUILD}/libBlasPlugin.so"

# Configure and build the plugin as it is in the repo. After an edit to a
# .td.txt or .cpp file, the rebuild cell further down is enough.
!cmake -S {PLUGIN_SRC} -B {PLUGIN_BUILD} -G Ninja \
    -DMLIR_DIR={INSTALL_DIR}/lib/cmake/mlir \
    -DLLVM_DIR={INSTALL_DIR}/lib/cmake/llvm \
    -DCMAKE_BUILD_TYPE=Release
assert _exit_code == 0
!ninja -C {PLUGIN_BUILD}
assert _exit_code == 0

-- Using MLIRConfig.cmake in: /content/mlir-tutorial-llvm/lib/cmake/mlir
-- Using LLVMConfig.cmake in: /content/mlir-tutorial-llvm/lib/cmake/llvm
-- Building with -fPIC
-- Configuring done (0.1s)
-- Generating done (0.0s)
-- Build files have been written to: /content/sc26_tutorial/session1/blas-dialect/build
ninja: Entering directory `/content/sc26_tutorial/session1/blas-dialect/build'
ninja: no work to do.


**[1.22]** With the plugin loaded, `mlir-opt` can round-trip `session1/matvec_custom.mlir`, which uses `blas.matvec`.

In [175]:
# [1.23]
MATVEC_CUSTOM_FILE = f"{REPO_DIR}/session1/matvec_custom.mlir"

!mlir-opt --load-dialect-plugin={PLUGIN_SO} {MATVEC_CUSTOM_FILE}
assert _exit_code == 0

module {
  func.func @matvec(%arg0: memref<4x3xf32>, %arg1: memref<3xf32>, %arg2: memref<4xf32>) {
    blas.matvec %arg0, %arg1, %arg2 : memref<4x3xf32>, memref<3xf32>, memref<4xf32>
    return
  }
}



**[1.24]** **Exercise 2: define a custom op, `blas.matmul`, that represents the loop nest from step 1 as a single operation.** The goal is an op that round-trips through `mlir-opt`: with the plugin loaded, `mlir-opt` reads `session1/matmul_custom.mlir` (printed by the next cell) and prints `blas.matmul` back in the same syntax, as it did for `blas.matvec` above. There are two parts: 2a, the op's ODS definition, and 2b, its verifier in C++. The references are the `matvec` versions, directly above each `TODO`.

In [176]:
# [1.25]
MATMUL_CUSTOM_FILE = f"{REPO_DIR}/session1/matmul_custom.mlir"

!cat {MATMUL_CUSTOM_FILE}

// The spec for the blas.matmul op: the definition in
// blas-dialect/include/Blas/BlasOps.td.txt is correct when mlir-opt can parse
// this file and print blas.matmul back in the same syntax.
func.func @matmul(%a: memref<4x3xf32>, %b: memref<3x5xf32>, %out: memref<4x5xf32>) {
  blas.matmul %a, %b, %out : memref<4x3xf32>, memref<3x5xf32>, memref<4x5xf32>
  return
}


**[1.26]** **2a. Define the op in ODS.** Ops are defined in ODS (Operation Definition Specification), a TableGen file describing an op's operands, types and syntax. The build runs `mlir-tblgen` on it to generate the op's C++ class. The next cell opens `session1/blas-dialect/include/Blas/BlasOps.td.txt`. Define `Blas_MatMulOp` where the `TODO` is, including `let hasVerifier = 1;` as `Blas_MatVecOp` does. (TableGen files are normally `.td`, but Colab's editor will not open that extension.)

In [177]:
# [1.27]
# Opens the ODS file. On Colab, files.view() puts it in the editor pane --
# use it instead of the file browser, which refuses some file types. Running
# locally there is no google.colab, so this prints the path instead, to
# open in an editor.

BLAS_DIALECT = f"{REPO_DIR}/session1/blas-dialect"

try:
    from google.colab import files

    files.view(f"{BLAS_DIALECT}/include/Blas/BlasOps.td.txt")   # define the op
except ModuleNotFoundError:
    print("open in an editor:")
    print(f"    {BLAS_DIALECT}/include/Blas/BlasOps.td.txt   (define the op)")

<IPython.core.display.Javascript object>

**[1.28]** **2b. Write the verifier in C++.** The ODS definition alone is not enough. It checks that each operand is a 2D `f32` memref, but it cannot say how their dimensions depend on each other. That check goes in `LogicalResult MatMulOp::verify()`, which `hasVerifier = 1` declares. The next cell opens `session1/blas-dialect/lib/BlasOps.cpp`; write `verify()` where the `TODO` is. `blas.matmul` has three constraints, on the reduction dimension, the rows and the columns. Each is one call to the `verifyDimsMatch` helper in that file. The helper lets a `?` dimension through, because its size is only known at run time.

In [178]:
# [1.29]
# Opens the verifier's file, as above.

try:
    from google.colab import files

    files.view(f"{BLAS_DIALECT}/lib/BlasOps.cpp")   # write the verifier
except ModuleNotFoundError:
    print("open in an editor:")
    print(f"    {BLAS_DIALECT}/lib/BlasOps.cpp   (write the verifier)")

<IPython.core.display.Javascript object>

**[1.30]** **Once 2a and 2b are both done, rebuild the plugin and round-trip `matmul_custom.mlir`.** Re-run these two cells after every edit. The op is defined when `blas.matmul` round-trips without errors.

- `custom op 'blas.matmul' is unknown`: the op isn't defined yet, or the plugin didn't rebuild.
- `undefined symbol: _ZN4blas8MatMulOp6verifyEv`, which is `blas::MatMulOp::verify()`: 2a declares the verifier but 2b isn't written yet.

In [179]:
# [1.31]
# Rebuild the plugin. Re-run this cell after EVERY edit to a .td.txt or .cpp
# file -- those are compiled into libBlasPlugin.so, unlike the .mlir files,
# which are read fresh each time. The assert matters: a failed build leaves
# the previous plugin in place, so the check below would keep testing the
# last working version.
!ninja -C {PLUGIN_BUILD}
assert _exit_code == 0

ninja: Entering directory `/content/sc26_tutorial/session1/blas-dialect/build'
ninja: no work to do.


In [180]:
# [1.32]
!mlir-opt --load-dialect-plugin={PLUGIN_SO} {MATMUL_CUSTOM_FILE}
assert _exit_code == 0

module {
  func.func @matmul(%arg0: memref<4x3xf32>, %arg1: memref<3x5xf32>, %arg2: memref<4x5xf32>) {
    blas.matmul %arg0, %arg1, %arg2 : memref<4x3xf32>, memref<3x5xf32>, memref<4x5xf32>
    return
  }
}



**[1.33]** The build ran `mlir-tblgen` on `BlasOps.td.txt` and turned it into C++, and that is where the names the next step uses come from: `getA()`, `getB()`, `getOut()` for the operands, the `build()` overloads a pattern calls to create the op, and the `verify()` from Exercise 2b. Nothing to edit here — it is regenerated on every build.

In [181]:
# [1.34]
import re

BLAS_OPS_HEADER = f"{PLUGIN_BUILD}/include/Blas/BlasOps.h.inc"
declaration = re.search(r"class MatMulOp :.*?\n\};", open(BLAS_OPS_HEADER).read(), re.S)
assert declaration, (
    "no MatMulOp in the generated header -- is the op defined in BlasOps.td.txt "
    "and the plugin rebuilt?")

print(f"{BLAS_OPS_HEADER}\n")
for line in declaration.group(0).split("\n"):
    if re.search(r"^class MatMulOp :|get[A-Z]\w*\(\)|static void build|LogicalResult verify\(", line):
        print(line.strip())

/content/sc26_tutorial/session1/blas-dialect/build/include/Blas/BlasOps.h.inc

class MatMulOp : public ::mlir::Op<MatMulOp, ::mlir::OpTrait::ZeroRegions, ::mlir::OpTrait::ZeroResults, ::mlir::OpTrait::ZeroSuccessors, ::mlir::OpTrait::NOperands<3>::Impl, ::mlir::OpTrait::OpInvariants> {
static ::llvm::ArrayRef<::llvm::StringRef> getAttributeNames() {
static constexpr ::llvm::StringLiteral getOperationName() {
return {std::next(getOperation()->operand_begin(), valueRange.first),
std::next(getOperation()->operand_begin(), valueRange.first + valueRange.second)};
::mlir::TypedValue<::mlir::MemRefType> getA() {
::mlir::TypedValue<::mlir::MemRefType> getB() {
::mlir::TypedValue<::mlir::MemRefType> getOut() {
::mlir::OpOperand &getAMutable() {
return getOperation()->getOpOperand(range.first);
::mlir::OpOperand &getBMutable() {
return getOperation()->getOpOperand(range.first);
::mlir::OpOperand &getOutMutable() {
return getOperation()->getOpOperand(range.first);
return {std::next(getOperation()

## [1.35] Step 3 — lowering `blas` to loops

### Writing the lowering

**How the pass is put together.** `convert-blas-to-scf` lowers operations of the `blas` dialect into loops, using `scf.for` operations. It is in `session1/blas-dialect/lib/LowerBlasToSCF.cpp`.

- **Declaration.** The pass is declared in TableGen, like the op. `include/Blas/Passes.td` has `def LowerBlasToSCFPass : Pass<"convert-blas-to-scf", "::mlir::func::FuncOp">`, where `convert-blas-to-scf` is the name of the pass. `mlir-tblgen` generates a base class, `LowerBlasToSCFPassBase`. The pass implementation, `LowerBlasToSCFPass`, derives from this base class. `::mlir::func::FuncOp` makes it a function-level pass: `runOnOperation()`, the implementation of the pass, runs on each function (`func.func`) in the input program.
- **Patterns.** Each pattern rewrites one op. `MatVecToSCFPattern` derives from `OpRewritePattern<MatVecOp>`, which ties it to `blas.matvec`. Its `matchAndRewrite` builds the loops and erases the op.
- **`runOnOperation()`** adds the patterns with `patterns.add<...>`. It then hands them to `applyPatternsGreedily`, along with the function, and each pattern is applied to the ops of its kind in that function. A pattern that is not added never runs.

More: [declaring passes](https://mlir.llvm.org/docs/PassManagement/#declarative-pass-specification), [patterns and the rewrite driver](https://mlir.llvm.org/docs/PatternRewriter/).

**Exercise 3: lower `blas.matmul` to the loops from step 1, in `MatMulToSCFPattern`.** The reference is `MatVecToSCFPattern`, in the same file, and the file has a `TODO` at each spot.

- **Uncomment `MatMulToSCFPattern`.** Its scaffolding is already written: the operands, the index constants, the problem sizes and the `eraseOp` that retires the op.
- **Write the loop nest** where it says so. As in the reference, build each loop's body by moving the rewriter's insertion point into the loop's block. A loop with no `iter_args` gets its `scf.yield` created automatically. The reduction loop has `iter_args`, so it ends with an explicit `scf::YieldOp::create`.
- **Add the pattern** in `runOnOperation()`.

**Then run the cells below one at a time.** They open the file, rebuild the plugin, run the pass and check the result. After each edit, re-run them from the rebuild cell on.

In [182]:
# [1.36]
# Opens the files below. On Colab, files.view() puts them in the editor
# pane -- use it instead of the file browser, which refuses some file
# types. Running locally there is no google.colab, so this prints the
# paths instead, to open in an editor.

try:
    from google.colab import files

    files.view(f"{BLAS_DIALECT}/lib/LowerBlasToSCF.cpp")   # pattern + registration
except ModuleNotFoundError:
    print("open in an editor:")
    print(f"    {BLAS_DIALECT}/lib/LowerBlasToSCF.cpp   (pattern + registration)")

<IPython.core.display.Javascript object>

In [183]:
# [1.37]
# Rebuild the plugin. Re-run this cell after EVERY edit to a .td.txt or .cpp
# file -- those are compiled into libBlasPlugin.so, unlike the .mlir files,
# which are read fresh each time. The assert matters: a failed build leaves
# the previous plugin in place, so the checks below would keep testing the
# last working version.
!ninja -C {PLUGIN_BUILD}
assert _exit_code == 0

ninja: Entering directory `/content/sc26_tutorial/session1/blas-dialect/build'
ninja: no work to do.


**[1.38]** Run the pass, writing the result to a file.

In [184]:
# [1.39]
LOWERED_MATMUL_FILE = "matmul_custom_lowered.mlir"

!mlir-opt --load-dialect-plugin={PLUGIN_SO} --load-pass-plugin={PLUGIN_SO} "--pass-pipeline=builtin.module(func.func(convert-blas-to-scf))" {MATMUL_CUSTOM_FILE} -o {LOWERED_MATMUL_FILE}
assert _exit_code == 0
!cat {LOWERED_MATMUL_FILE}

module {
  func.func @matmul(%arg0: memref<4x3xf32>, %arg1: memref<3x5xf32>, %arg2: memref<4x5xf32>) {
    %c0 = arith.constant 0 : index
    %c1 = arith.constant 1 : index
    %c4 = arith.constant 4 : index
    %c5 = arith.constant 5 : index
    %c3 = arith.constant 3 : index
    scf.for %arg3 = %c0 to %c4 step %c1 {
      scf.for %arg4 = %c0 to %c5 step %c1 {
        %0 = memref.load %arg2[%arg3, %arg4] : memref<4x5xf32>
        %1 = scf.for %arg5 = %c0 to %c3 step %c1 iter_args(%arg6 = %0) -> (f32) {
          %2 = memref.load %arg0[%arg3, %arg5] : memref<4x3xf32>
          %3 = memref.load %arg1[%arg5, %arg4] : memref<3x5xf32>
          %4 = arith.mulf %2, %3 : f32
          %5 = arith.addf %arg6, %4 : f32
          scf.yield %5 : f32
        }
        memref.store %1, %arg2[%arg3, %arg4] : memref<4x5xf32>
      }
    }
    return
  }
}



**[1.40]** Load the lowered file, run it through the same CPU pipeline as step 1, JIT it, and check the result. Compare the output above with `matmul_scf.mlir` from Exercise 1: the pass built the same loops that were written by hand there.

In [185]:
# [1.41]
custom_module = Module.parseFile(LOWERED_MATMUL_FILE, ctx)
PassManager.parse(CPU_LOWERING_PIPELINE, ctx).run(custom_module.operation)
run_and_check(custom_module, "matmul", [a, b, out], expected=a @ b)

out      = [[ 25.  28.  31.  34.  37.]
 [ 70.  82.  94. 106. 118.]
 [115. 136. 157. 178. 199.]
 [160. 190. 220. 250. 280.]]
expected = [[ 25.  28.  31.  34.  37.]
 [ 70.  82.  94. 106. 118.]
 [115. 136. 157. 178. 199.]
 [160. 190. 220. 250. 280.]]
OK


array([[ 25.,  28.,  31.,  34.,  37.],
       [ 70.,  82.,  94., 106., 118.],
       [115., 136., 157., 178., 199.],
       [160., 190., 220., 250., 280.]], dtype=float32)